# 21 — Direction 1, V2: train through the replica (SHAC), score, package for submission

One GPU T4 ×2 session (`hvac_controller/d1/train.py`, `policy.py`, `evaluate.py`, `augment.py`):

1. **setup:** the held-out scenarios' folders are rebuilt from the dataset's building YAMLs and weather presets (for
   the original simulator).
2. **smoke** (≈ 15 min, GPU 0): stops the notebook if anything fails.
   - parity of the submission path (`KagglePolicy` on the organisers' simulator, CPU) with the training path (GPU,
     in a padded batch with another building);
   - the padded multi-building environment against each building alone;
   - checkpointed against plain gradients;
   - then a short run of the full configuration, to measure seconds per update and memory.
3. **training**, both GPUs at once, `MINUTES` each. Both runs use the same data and settings and differ in one thing:
   - `main` (GPU 0): V2, with the weather and zone-temperature memory;
   - `no_memory` (GPU 1): the same without it (the ablation).
   - Training set: the 25 shipped scenarios outside building_train_2 and the 180 generated ones.
   - Held out: building_train_2's 5, building_train_2 × minneapolis_1/_5, and 6 generated buildings × 2 strips of
     held-out stations (19 scenarios).
   - Each run keeps its best checkpoint on the held-out replica score.
4. **winner** = the run with the lower held-out replica score (30 days, the averaged weights). It is scored in the
   **original simulator** (4 CPUs) on 13 held-out scenarios: building_train_2's 5 shipped ones, the first strip of
   each held-out building, and the 2 minneapolis ones. It is compared with K10 and V1 (from notebook 20).
5. **package:**
   - `/kaggle/working/data.zip`, flat: `d1_policy.pt`, `d1_policy.py`, `README.txt`. Upload it as a **new** Kaggle
     dataset (keep V1's); notebook 22 submits it.
   - `/kaggle/working/d1_train_v2.zip`: the logs.

**Setup:** accelerator **GPU T4 ×2**; Internet on (clones the repo). Attach the competition data and the V2 dataset
(`data/data.zip`: 224 replica caches, building YAMLs, held-out weather presets, manifest, K10 and V1 results).
≈ 8.5 h with `MINUTES = 420`.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
HELD_OUT = "building_train_2"
SETTINGS = {"h": 288, "gamma": 0.998, "seg": 24, "buckets": 3, "winsor": 10.0, "spike": 10.0, "ema": 0.98,
            "eval_minutes": 20, "eval_seeds": 1}
RUNS = {"main": {"memory": 1}, "no_memory": {"memory": 0}}
MINUTES = 420             # training budget of each run

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

import torch
N_GPU = torch.cuda.device_count()
assert N_GPU >= 2, "set the accelerator to GPU T4 x2"
print(torch.__version__, N_GPU, "x", torch.cuda.get_device_name())

REPO = Path("/tmp/hvac-controller")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
SHIPPED = next(p for p in sorted(Path("/kaggle/input").rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
ROOT = REPO / "smart_buildings" / "training_scenarios"
shutil.copytree(SHIPPED, ROOT, dirs_exist_ok=True)
sys.path.insert(0, str(REPO))
ENV = dict(os.environ, PYTHONPATH=str(REPO), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")

DATA = next(p.parent for p in sorted(Path("/kaggle/input").rglob("manifest_v2.json")))
MANIFEST = json.loads((DATA / "manifest_v2.json").read_text())
CACHES = {p.stem: p for p in sorted(DATA.glob("*.pkl"))}
for p in DATA.glob("building_0*.yaml"):
    shutil.copy(p, REPO / "smart_buildings" / "buildings" / p.name)
(REPO / "smart_buildings" / "weather" / "csv").mkdir(parents=True, exist_ok=True)
for p in DATA.glob("x*.yaml"):
    shutil.copy(p, REPO / "smart_buildings" / "weather" / p.name)
for p in DATA.glob("x*.csv"):
    shutil.copy(p, REPO / "smart_buildings" / "weather" / "csv" / p.name)
VAL_TAGS = "gen_val,shipped_building_new_station"
r = subprocess.run([sys.executable, "-m", "hvac_controller.d1.augment", "scenarios", str(DATA / "manifest_v2.json"),
                    "--tags", VAL_TAGS], cwd=REPO, env=ENV, capture_output=True, text=True)
print("held-out scenario folders:", r.returncode, (r.stdout + r.stderr)[-1500:])
assert r.returncode == 0

SHIPPED_ALL = sorted(f"{p.parent.parent.name}__{p.parent.name}" for p in SHIPPED.glob("*/*/config.gin"))
TRAIN = [s for s in SHIPPED_ALL if not s.startswith(HELD_OUT + "__")] + [e["label"] for e in MANIFEST if e["tag"] == "gen_train"]
VAL = [s for s in SHIPPED_ALL if s.startswith(HELD_OUT + "__")] + [e["label"] for e in MANIFEST if e["tag"] in VAL_TAGS.split(",")]
first_strip = {}
for e in MANIFEST:
    if e["tag"] == "gen_val":
        first_strip.setdefault(e["building"], e["label"])
VAL_ORIGINAL = ([s for s in SHIPPED_ALL if s.startswith(HELD_OUT + "__")] + sorted(first_strip.values())
                + [e["label"] for e in MANIFEST if e["tag"] == "shipped_building_new_station"])
missing = [s for s in TRAIN + VAL if s not in CACHES]
assert not missing, f"replica caches missing (attach the V2 data.zip dataset): {missing[:10]}"
REFS = [json.loads(p.read_text()) for p in sorted(DATA.glob("original__*__s0.json"))]
print(f"{len(TRAIN)} training, {len(VAL)} held-out scenarios ({len(VAL_ORIGINAL)} for the original simulator); "
      f"{len(CACHES)} caches, {len(REFS)} reference runs")
OUT = Path("/kaggle/working/d1_train_v2")
OUT.mkdir(parents=True, exist_ok=True)


def train_cmd(out_dir, settings, extra=()):
    cmd = [sys.executable, "-m", "hvac_controller.d1.train", str(out_dir),
           "--caches", ",".join(str(CACHES[s]) for s in TRAIN), "--val-caches", ",".join(str(CACHES[s]) for s in VAL)]
    for k, v in settings.items():
        cmd += [f"--{k.replace('_', '-')}", str(v)]
    return cmd + list(extra)


def tail(path, n=3000):
    text = Path(path).read_text() if Path(path).exists() else ""
    return text[-8000:] if "Traceback" in text else text[-n:]

In [ ]:
# 1. Smoke: start-up checks and a short run of the full configuration on GPU 0. Stops here if anything is wrong.
smoke = OUT / "smoke"
smoke.mkdir(exist_ok=True)
t0 = time.time()
with open(smoke / "log.txt", "w") as log:
    rc = subprocess.run(train_cmd(smoke, {**SETTINGS, "minutes": 4, "eval_minutes": 1000, "eval_steps": 288}, ["--smoke"]),
                        cwd=REPO, env=dict(ENV, CUDA_VISIBLE_DEVICES="0"), stdout=log, stderr=subprocess.STDOUT).returncode
print(f"smoke exit {rc} in {time.time() - t0:.0f}s\n" + tail(smoke / "log.txt", 6000))
rep = json.loads((smoke / "report.json").read_text()) if (smoke / "report.json").exists() else {}
assert rc == 0 and all(rep.get(k, {}).get("ok") for k in ("parity", "group", "checkpoint")), "smoke failed: read the log"
rows = [json.loads(l) for l in (smoke / "train_log.jsonl").read_text().splitlines()]
s_it = sum(r["s"] for r in rows[1:]) / max(len(rows) - 1, 1) if len(rows) > 1 else rows[0]["s"]
print(f"{len(rows)} updates, {s_it:.0f} s per update, peak {max(r['mem_gb'] for r in rows):.1f} GB: "
      f"about {MINUTES * 60 / s_it:.0f} updates in {MINUTES} min (want >= 150)")

In [ ]:
# 2. The two runs, one per GPU; evaluations, skipped spikes and rollbacks are echoed as they happen.
t0, procs = time.time(), {}
for gpu, (name, extra) in enumerate(RUNS.items()):
    d = OUT / name
    d.mkdir(exist_ok=True)
    log = open(d / "log.txt", "w")
    procs[name] = (subprocess.Popen(train_cmd(d, {**SETTINGS, **extra, "minutes": MINUTES}), cwd=REPO, stdout=log,
                                    stderr=subprocess.STDOUT, env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))), log, d)
    print(f"GPU {gpu}: {name} {extra}", flush=True)
seen = {name: 0 for name in procs}
while True:
    running = any(p.poll() is None for p, _, _ in procs.values())
    for name, (p, _, d) in procs.items():
        lines = (d / "log.txt").read_text().splitlines()
        for line in lines[seen[name]:]:
            if line.startswith(("[eval]", "Traceback")) or "restored" in line or "skipped" in line or "bucket(s)" in line:
                print(f"[{(time.time() - t0) / 60:5.0f} min] {name}: {line[:400]}", flush=True)
        seen[name] = len(lines)
    if not running:
        break
    time.sleep(60)
REPORTS = {}
for name, (p, log, d) in procs.items():
    log.close()
    REPORTS[name] = json.loads((d / "report.json").read_text()) if (d / "report.json").exists() else {}
    print(f"===== {name}: exit {p.returncode}, best held-out {REPORTS[name].get('best')}, "
          f"{REPORTS[name].get('iterations')} updates, {REPORTS[name].get('spikes')} skipped spikes, "
          f"{REPORTS[name].get('rollbacks')} rollbacks\n" + tail(d / "log.txt", 2500))
done = {n: r["best"] for n, r in REPORTS.items() if r.get("best") is not None and (OUT / n / "d1_policy.pt").exists()}
WINNER = min(done, key=done.get)
print("winner:", WINNER, done)

In [ ]:
# 3. The organisers' simulator: the winner on 13 held-out scenarios, against K10 and V1 (notebook 20).
from hvac_controller.d1 import evaluate as ev

os.environ.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", CUDA_VISIBLE_DEVICES="",
                  PYTHONPATH=str(REPO))
os.chdir(REPO)
EVAL = OUT / "original"
EVAL.mkdir(exist_ok=True)
for p in sorted(DATA.glob("original__*__s0.json")):
    shutil.copy(p, EVAL / p.name)
policy = OUT / WINNER / "d1_policy.pt"
jobs = [{"where": "original", "scenario": s.replace("__", "/", 1), "policy": f"trained:{policy}", "seed": 0,
         "steps": 8640, "config": str(ROOT / s.replace("__", "/", 1) / "config.gin"), "cache": None} for s in VAL_ORIGINAL]
results = ev.run_jobs(jobs, EVAL, 4)
ens = ev.ensembles(results)
SUMMARY = {}
for ref in ("k10", "v1"):
    cmp = ev.compare(ens, WINNER, ref)
    SUMMARY[ref] = cmp.get("mean")
    print(f"\n{WINNER} in the original vs {ref}:")
    for scenario, c in cmp.items():
        if scenario == "mean":
            print(f"  mean over {len(cmp) - 1}: {c['candidate']:.5f} vs {ref} {c['null']:.5f} ({c['candidate'] / c['null'] - 1:+.1%})")
            continue
        e = ens[("original", scenario, WINNER)]
        print(f"  {scenario:34s} {c['candidate']:.5f} vs {c['null']:.5f} ({c['rel']:+.1%}) | "
              f"regret {e['regret']:.4f} cost {e['cost']:.4f} carbon {e['carbon']:.4f}")
print("\nGATE V2: winner below V1 on the held-out scenarios in the original:",
      SUMMARY.get("v1") is not None and SUMMARY["v1"]["candidate"] < SUMMARY["v1"]["null"])

In [ ]:
# 4. Package the submission dataset (flat data.zip) and the logs.
readme = (f"Direction 1 V2 policy ({WINNER}), commit {COMMIT}: trained on {len(TRAIN)} scenarios, chosen on {len(VAL)} "
          f"held-out ones (replica {done[WINNER]:.5f}); original simulator on {len(VAL_ORIGINAL)} held-out scenarios: "
          f"{json.dumps(SUMMARY)}. d1_policy.py = hvac_controller/d1/policy.py. Used by notebook 22.\n")
with zipfile.ZipFile("/kaggle/working/data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(policy, "d1_policy.pt")
    z.write(REPO / "hvac_controller" / "d1" / "policy.py", "d1_policy.py")
    z.writestr("README.txt", readme)
print(readme, "wrote /kaggle/working/data.zip:", zipfile.ZipFile("/kaggle/working/data.zip").namelist())
name = "/kaggle/working/d1_train_v2.zip"
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and (p.suffix != ".pt" or p.name == "d1_policy.pt"):
            z.write(p, p.relative_to(OUT.parent))
print("wrote", name)